# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samanashfaq05/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!git clone https://github.com/samanashfaq05/flyrank-ml-internship.git
%cd /content/flyrank-ml-internship

!pip install -q datasets

from datasets import load_dataset
import pandas as pd

ds = load_dataset(
    "FlyRank/internship-lanes",
    "decline_recovery"
)

print(ds)

for split in ds.keys():
    df = ds[split].to_pandas()

    print("\n" + "=" * 70)
    print("SPLIT:", split)
    print("Rows:", len(df))
    print("\nCOLUMNS:")
    print(df.columns.tolist())

    print("\nFIRST 5 ROWS:")
    print(df.head().to_string())

    print("\nDATA TYPES:")
    print(df.dtypes)

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 195, done.
remote: Counting objects: 100% (195/195), done.
remote: Compressing objects: 100% (143/143), done.
remote: Total 195 (delta 93), reused 101 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (195/195), 1.88 MiB | 5.12 MiB/s, done.
Resolving deltas: 100% (93/93), done.
/content/flyrank-ml-internship


README.md:   0%|          | 0.00/2.98k [00:00<?, ?B/s]

default_lanes/decline_recovery.parquet: reconstructing file:   0%|          |  0.00B / 3.88MB            

default_lanes/decline_recovery.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/56253 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'keyword_char_count', 'keyword_token_count', 'public_url_hash_id', 'public_url_char_count', 'public_url_path_depth', 'content_title_hash_id', 'content_title_char_count', 'content_title_token_count', 'impressions_90d', 'clicks_90d', 'sum_position_90d', 'sessions_90d', 'pageviews_90d', 'ai_sessions_90d', 'scroll_events_90d', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'client_has_gsc', 'client_has_ga4', 'content_type', 'content_created_at', 'content_age_days', 'ctr_90d', 'avg_position_90d', 'ai_traffic_pct_90d', 'scroll_rate_90d', 'trend_direction', 'trend_pct', 'health_score', 'needs_indexing', 'is_quick_win', 'needs_ctr_fix', 'needs_engagement_fix', 'ai_opportunity', 'is_underperformer', 'is_declining'],
        num_rows: 56253
    })
})

SPLIT: train
Rows: 56253

COLUMNS:
['client_hash_id', 'co

In [3]:
df = ds["train"].to_pandas().copy()

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Clients:", df["client_hash_id"].nunique())
print("Declining rate:", round(df["is_declining"].mean(), 3))

Rows: 56253
Columns: 43
Clients: 46
Declining rate: 1.0


## 1. Question

*The research question and the decision it supports.*

# Problem framing

## Research question

Among content records that are already experiencing declining search performance, can observable content characteristics and historical performance signals help identify records experiencing severe decline, so editors can prioritize them for review and recovery?

## Decision supported

The analysis supports the decision of which declining content records should be investigated first.

## Unit of analysis

Each row represents a content record.

## Output

The model will produce a probability-based severe-decline score that can be used to rank content records for editor review.

## Action

A FlyRank editor could use the ranked output to investigate high-priority records and decide whether content refresh, SEO review, CTR improvements, or another recovery action is appropriate.

## Cost of wrong calls

A false positive may cause unnecessary editor review.

A false negative may cause a severely declining content record to receive attention too late.

## Why ML helps

The decision depends on multiple interacting signals, including historical search performance, traffic, engagement, content characteristics, and age. These relationships may be difficult to represent with a single hand-written rule, so machine learning may help prioritize records for human review.

## Claim boundary

This project provides decision-support based on observed patterns in the dataset. It does not prove why search performance declined and does not predict Google's algorithm.

In [7]:
import numpy as np

# Calculate observed percentage change in impressions
df["impression_change_pct"] = (
    (
        df["impressions_last_30d"]
        - df["impressions_prev_30d"]
    )
    / df["impressions_prev_30d"]
) * 100

# Define severe decline as a drop of 60% or more
df["severe_decline"] = (
    df["impression_change_pct"] <= -60
).astype(int)

print("CAPSTONE TARGET")
print("=" * 70)

print("Target: severe_decline")
print("Definition: impression change <= -60%")

print("\nSevere decline records:", df["severe_decline"].sum())
print(
    "Severe decline rate:",
    round(df["severe_decline"].mean(), 3)
)

print("\nNon-severe decline records:")
print((df["severe_decline"] == 0).sum())

CAPSTONE TARGET
Target: severe_decline
Definition: impression change <= -60%

Severe decline records: 26093
Severe decline rate: 0.464

Non-severe decline records:
30160


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

# Data safety

The dataset contains pseudonymous identifiers, performance metrics, content characteristics, and derived recommendation fields.

I use observable content characteristics and historical performance signals as candidate features.

I deliberately exclude all pseudonymous identifiers:

- client_hash_id
- content_hash_id
- keyword_hash_id
- public_url_hash_id
- content_title_hash_id

These columns are identifiers rather than meaningful predictive features. `client_hash_id` will only be used for grouped train/test splitting.

I also exclude label-derived fields:

- trend_direction
- trend_pct
- is_declining

These fields directly describe decline and could cause target leakage.

The target `severe_decline` is derived from the percentage change between `impressions_prev_30d` and `impressions_last_30d`. Therefore, those two columns and `impression_change_pct` are excluded from model features to prevent direct target leakage.

I also exclude derived recommendation or outcome-style fields:

- health_score
- needs_indexing
- is_quick_win
- needs_ctr_fix
- needs_engagement_fix
- ai_opportunity
- is_underperformer

These fields may contain business rules or downstream assessments rather than independent observable signals.

The model will therefore use only selected observable performance, traffic, engagement, content, and structural characteristics available independently of the severe-decline target.

No client-identifying information will be used as a model feature or included in the final recommendations.

In [8]:
# ============================================================
# SECTION 2: DATA SAFETY AND FEATURE SELECTION
# ============================================================

# Pseudonymous identifier columns
id_cols = [
    "client_hash_id",
    "content_hash_id",
    "keyword_hash_id",
    "public_url_hash_id",
    "content_title_hash_id",
]

# Columns directly related to decline or used to define the target
leakage_cols = [
    "trend_direction",
    "trend_pct",
    "is_declining",
    "impression_change_pct",
    "impressions_prev_30d",
    "impressions_last_30d",
]

# Derived recommendation / outcome-style columns
derived_cols = [
    "health_score",
    "needs_indexing",
    "is_quick_win",
    "needs_ctr_fix",
    "needs_engagement_fix",
    "ai_opportunity",
    "is_underperformer",
]

# Other non-feature columns
non_feature_cols = [
    "content_created_at",
    "severe_decline"
]

# Combine all excluded columns
excluded_cols = (
    id_cols
    + leakage_cols
    + derived_cols
    + non_feature_cols
)

# Keep only columns that actually exist
excluded_cols = [
    col for col in excluded_cols
    if col in df.columns
]

# Candidate model features
feature_cols = [
    col for col in df.columns
    if col not in excluded_cols
]

print("DATA SAFETY CHECK")
print("=" * 70)

print("\nTotal columns:", len(df.columns))

print("\nExcluded columns:", len(excluded_cols))
for col in excluded_cols:
    print("-", col)

print("\nCandidate model features:", len(feature_cols))
for col in feature_cols:
    print("-", col)

print("\nIdentifier columns used as features:")
print(any(col in feature_cols for col in id_cols))

print("\nLeakage columns used as features:")
print(any(col in feature_cols for col in leakage_cols))

DATA SAFETY CHECK

Total columns: 45

Excluded columns: 20
- client_hash_id
- content_hash_id
- keyword_hash_id
- public_url_hash_id
- content_title_hash_id
- trend_direction
- trend_pct
- is_declining
- impression_change_pct
- impressions_prev_30d
- impressions_last_30d
- health_score
- needs_indexing
- is_quick_win
- needs_ctr_fix
- needs_engagement_fix
- ai_opportunity
- is_underperformer
- content_created_at
- severe_decline

Candidate model features: 25
- keyword_char_count
- keyword_token_count
- public_url_char_count
- public_url_path_depth
- content_title_char_count
- content_title_token_count
- impressions_90d
- clicks_90d
- sum_position_90d
- sessions_90d
- pageviews_90d
- ai_sessions_90d
- scroll_events_90d
- clicks_last_30d
- sessions_last_30d
- clicks_prev_30d
- sessions_prev_30d
- client_has_gsc
- client_has_ga4
- content_type
- content_age_days
- ctr_90d
- avg_position_90d
- ai_traffic_pct_90d
- scroll_rate_90d

Identifier columns used as features:
False

Leakage columns

In [9]:
# ============================================================
# FEATURE TYPE CHECK
# ============================================================

feature_df = df[feature_cols].copy()

print("FEATURE DATA TYPES")
print("=" * 70)

print(feature_df.dtypes.value_counts())

categorical_features = feature_df.select_dtypes(
    include=["object", "bool"]
).columns.tolist()

numeric_features = feature_df.select_dtypes(
    include=["number"]
).columns.tolist()

print("\nNumeric features:", len(numeric_features))
print(numeric_features)

print("\nCategorical / boolean features:", len(categorical_features))
print(categorical_features)

print("\nMissing values:")
missing = feature_df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

FEATURE DATA TYPES
int64      14
float64     8
bool        2
object      1
Name: count, dtype: int64

Numeric features: 22
['keyword_char_count', 'keyword_token_count', 'public_url_char_count', 'public_url_path_depth', 'content_title_char_count', 'content_title_token_count', 'impressions_90d', 'clicks_90d', 'sum_position_90d', 'sessions_90d', 'pageviews_90d', 'ai_sessions_90d', 'scroll_events_90d', 'clicks_last_30d', 'sessions_last_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'ctr_90d', 'avg_position_90d', 'ai_traffic_pct_90d', 'scroll_rate_90d']

Categorical / boolean features: 3
['client_has_gsc', 'client_has_ga4', 'content_type']

Missing values:
ai_traffic_pct_90d    14896
scroll_rate_90d       14849
pageviews_90d         13598
sessions_90d          13598
scroll_events_90d     13598
ai_sessions_90d       13598
dtype: int64


### Data safety observations

After excluding identifiers, label-derived fields, target-related columns, and derived recommendation fields, 25 candidate features remain: 22 numeric features and 3 categorical or boolean features.

Several traffic and engagement features contain missing values. Instead of removing a large number of records, missing numeric values will be imputed during preprocessing. The categorical feature `content_type` will be encoded without using pseudonymous identifiers.

The grouped client identifier will be retained only for the train/test split and will never be used as a model feature.

This preprocessing design aims to prevent direct target leakage while preserving useful observable information.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

# Transparent baseline

Before training a machine learning model, I build a simple rule-based baseline that a human can understand.

The baseline prioritizes a content record when:

1. Its recent impressions are lower than its previous impressions.
2. Its average search position is relatively weak.
3. It previously had meaningful search visibility.

Each condition contributes one point to a transparent priority score. The records are ranked by this score.

This baseline is intentionally simple. The Random Forest model will only be useful if it can improve prioritization compared with this transparent rule on the same grouped holdout data and using the same Precision@50 metric.

In [10]:
from sklearn.model_selection import GroupShuffleSplit

# Features
X = df[feature_cols].copy()

# Target
y = df["severe_decline"].copy()

# Grouping variable - used ONLY for splitting
groups = df["client_hash_id"]

# Client-level split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("GROUPED TRAIN/TEST SPLIT")
print("=" * 70)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Clients appearing in both:", len(train_clients & test_clients))

print("\nTraining severe decline rate:", round(y_train.mean(), 3))
print("Test severe decline rate:", round(y_test.mean(), 3))

GROUPED TRAIN/TEST SPLIT
Training rows: 48219
Test rows: 8034

Training clients: 36
Test clients: 10
Clients appearing in both: 0

Training severe decline rate: 0.454
Test severe decline rate: 0.524


In [11]:
import numpy as np

# ============================================================
# TRANSPARENT RULE-BASED BASELINE
# ============================================================

baseline_df = df.iloc[test_idx].copy()

# Rule 1:
# Recent impressions are lower than previous impressions
recent_decline = (
    baseline_df["impressions_last_30d"]
    < baseline_df["impressions_prev_30d"]
).astype(int)

# Rule 2:
# Weak search position
weak_position = (
    baseline_df["avg_position_90d"] > 20
).astype(int)

# Rule 3:
# Previously meaningful visibility
meaningful_visibility = (
    baseline_df["impressions_prev_30d"] >= 500
).astype(int)

# Transparent baseline score
baseline_df["baseline_score"] = (
    recent_decline
    + weak_position
    + meaningful_visibility
)

# Reason codes
def get_reason(row):
    reasons = []

    if row["impressions_last_30d"] < row["impressions_prev_30d"]:
        reasons.append("recent_impression_drop")

    if row["avg_position_90d"] > 20:
        reasons.append("weak_position")

    if row["impressions_prev_30d"] >= 500:
        reasons.append("meaningful_visibility")

    return "|".join(reasons)

baseline_df["reason_code"] = baseline_df.apply(
    get_reason,
    axis=1
)

# Rank pages
baseline_df = baseline_df.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

baseline_df["rank"] = baseline_df.index + 1

print("BASELINE SCORE DISTRIBUTION")
print("=" * 70)

print(baseline_df["baseline_score"].value_counts().sort_index())

print("\nTOP 10 BASELINE RECOMMENDATIONS")
print("=" * 70)

print(
    baseline_df[
        [
            "rank",
            "baseline_score",
            "reason_code",
            "impressions_prev_30d",
            "impressions_last_30d",
            "avg_position_90d",
            "severe_decline"
        ]
    ].head(10).to_string(index=False)
)

BASELINE SCORE DISTRIBUTION
baseline_score
1    2310
2    4421
3    1303
Name: count, dtype: int64

TOP 10 BASELINE RECOMMENDATIONS
 rank  baseline_score                                                reason_code  impressions_prev_30d  impressions_last_30d  avg_position_90d  severe_decline
    1               3 recent_impression_drop|weak_position|meaningful_visibility                   630                    69              21.2               1
    2               3 recent_impression_drop|weak_position|meaningful_visibility                   577                   353              30.7               0
    3               3 recent_impression_drop|weak_position|meaningful_visibility                   609                   353              23.1               0
    4               3 recent_impression_drop|weak_position|meaningful_visibility                   576                    67              46.4               1
    5               3 recent_impression_drop|weak_position|meaningful_vis

In [13]:
# ============================================================
# BASELINE EVALUATION
# ============================================================

import numpy as np

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    top_k_labels = np.asarray(labels)[order[:k]]
    return top_k_labels.mean()

k = 50

baseline_precision_at_50 = precision_at_k(
    test_df["baseline_score"],
    test_df["severe_decline"],
    k=k
)

test_base_rate = test_df["severe_decline"].mean()

print("BASELINE EVALUATION")
print("=" * 70)

print(f"Precision@{k}:",
      round(baseline_precision_at_50, 3))

print("Test severe decline base rate:",
      round(test_base_rate, 3))

print("Random selection reference:",
      round(test_base_rate, 3))

NameError: name 'test_df' is not defined

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
